# V3 shorter screen: seeds 42 and 45
Stop the original V3 training cell first. This keeps all four preprocessing methods and all 12 participants, while screening two fixed seeds (stable 42 and collapse-prone 45). The total is 96 fits. Compatible completed fits are copied from the original run; only missing fits train. The original results stay in their original folder.
This is an exploratory two-seed screen. Additional seeds can follow if the results justify them. The training code, epoch limit, model, and early stopping are identical to full V3.

## 1. Setup
Use a GPU runtime. This notebook uses the fixed Git tag `v3-short-screen-v1` so reconnecting does not silently change the code.

In [ ]:
import os, sys, subprocess, json
from google.colab import drive
drive.mount('/content/drive')
repo = '/content/teco-v3-short'
run_ref = 'v3-short-screen-v1'
if not os.path.isdir(repo):
    subprocess.run(['git', 'clone', '-q', '--branch', run_ref, '--single-branch',
                    'https://github.com/parmisbathayan/reproduce-teco-results.git', repo], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
tag_commit = subprocess.check_output(['git', 'rev-parse', run_ref + '^{commit}'], text=True).strip()
assert commit == tag_commit, 'Use the fixed screening revision.'
assert not subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip(), 'The code was edited; start with a clean screening checkout.'
print('Screening Git commit:', commit)


## 2. Check and reuse completed work
This cell checks code, data, package versions, and copied files before creating a separate screening result folder. It performs no training.

In [ ]:
trt_dir = '/content/drive/MyDrive/Thesis/Data/TRT_Total'
labels_csv = '/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv'
base = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly'
source_out_dir = os.path.join(base, 'v3_preprocessing')
out_dir = os.path.join(base, 'v3_preprocessing_screen_42_45')
command = [sys.executable, '-B', 'src/v3_short_screen.py', '--trt-dir', trt_dir,
           '--labels-csv', labels_csv, '--source-out-dir', source_out_dir,
           '--out-dir', out_dir, '--config', 'configs/v3_short_screen.json']
subprocess.run(command + ['--prepare-only'], check=True)


## 3. Run or resume the missing fits
Each completed fit saves immediately. After a disconnect, rerun the setup, preparation, and training cells. Keep this output directory and configuration unchanged.

In [ ]:
subprocess.run(command, check=True)


## 4. Review the paired two-seed comparison
All four methods in this table use the same two seeds and all 12 participants. The original full-run raw baseline, which has five seeds, remains available separately.

In [ ]:
import pandas as pd
from IPython.display import display
metrics = pd.read_csv(os.path.join(out_dir, 'fold_metrics.csv'))
print('Screening fits saved:', len(metrics), '/ 96')
display(pd.read_csv(os.path.join(out_dir, 'summary_by_method.csv')))
display(pd.read_csv(os.path.join(out_dir, 'summary_by_method_seed.csv')))
